<a href="https://colab.research.google.com/github/LucasPuertas95/Procesamiento-del-Habla/blob/main/TP5_PUERTAS_LUCAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP5 CHATBOT RAG Y EVALUACION DE EMBBEDINGS

# PUERTAS LUCAS

# antiguo chat bot del TP4

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

In [ ]:
!pip install spacy --quiet
!python -m spacy download es_core_news_sm --quiet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# chatbot: FUEGIN (para la sanwucheria "Las Brasas")

Mi chat bot se llama fuegin y se aplica para la atencion al cliente de un establecimiento de comida "Sanwucheria Las Brasas", la funcion del chatbot es responder a las preguntas del cliente cuando este mismo necesita ver la oferta de productos ofrecidos y cuando necesita tomar su orden. el chatbot seria aplicable para caulquier otro establecimiento de este tipo

# Elementos de Preguntas y respuestas (pares):

# Chatbot FUEGIN con TF-IDF similitud del coseno

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

preguntas = [
    "Hola buenas tardes quiero pedir comida / Empezar",
    "Ver Menu Completo / Que opciones tienen para comer",
    "Mostrar la seccion de Sandwiches de Milanesa",
    "Mostrar la seccion de Sandwiches de Lomo",
    "Mostrar la seccion de Hamburguesas Caseras",
    "Mostrar la seccion de Sandwiches Vegetarianos",
    "Que ingredientes trae el Mila Simple y precio",
    "Que ingredientes trae el Mila Brasa Completo y precio",
    "Que ingredientes trae el Super Sanguchon de Milanesa",
    "Que ingredientes trae el Lomo Clasico y precio",
    "Que ingredientes trae el Lomo Fuego Sagrado y precio",
    "Que ingredientes trae la Burguer Simple y precio",
    "Que ingredientes trae la Bacon Brasa Burguer y precio",
    "Que ingredientes trae el Sandwich Veggie Brasa y precio",
    "Que ingredientes trae la Burguer de Garbanzo y precio",
    "Quiero encargar un sandwich / Como tomo el pedido",
    "Puedo sacar un ingrediente o armar a mi gusto el sandwich",
    "Tienen promociones o combos disponibles hoy",
    "Cual es el precio de las papas fritas y bebidas",
    "Confirmar Lista de Pedido / Ya elegi lo que quiero",
    "Retiro por el Local / Voy a buscar el pedido",
    "Enviar a Domicilio / Quiero delivery para mi casa",
    "Datos de Pago y Alias / Como puedo pagar el pedido",
    "Enviar Comprobante / Ya realice la transferencia del dinero",
    "Pedido en Cocina / Cuando esta listo mi sandwich",
    "Cancelar Pedido / Quiero dar de baja lo que pedi",
    "Horarios y Delivery / Que dias y a que hora abren",
    "Donde queda ubicado el local de Las Brasas",
    "Cuáles son los medios de pago en el local",
    "Tienen sandwiches para celiacos o sin TACC",
    "Venden postres para el final de la cena",
    "Llamar Soporte Humano / Hablar con una persona"
]

respuestas = [
    "Buenas! Bienvenido a Sangucheria Las Brasas, soy FUEGIN. Para arrancar, escribe 'Ver Menu Completo' para conocer nuestras categorias, o 'Horarios y Delivery' si quieres saber si llegamos a tu casa.",
    "En Las Brasas la especialidad es el fuego. Contamos con 4 categorias principales de sandwiches. Escribe el nombre de la categoria que te tiente para ver el detalle y precios: 1) 'Sandwiches de Milanesa', 2) 'Sandwiches de Lomo', 3) 'Hamburguesas Caseras', 4) 'Sandwiches Vegetarianos'.",
    "Los reyes del local! Tenemos estas opciones de milanesa (carne o pollo): 'Mila Simple' ($4.000), 'Mila Brasa Completo' ($5.200) o el 'Super Sanguchon de Milanesa' ($8.500). Dime el nombre de cualquiera de ellos (ej: 'Que trae el Mila Brasa') para ver sus ingredientes.",
    "Carne tierna y asada al momento! Nuestras opciones de lomo son: 'Lomo Clasico' ($4.800) y 'Lomo Fuego Sagrado' ($6.000). Escribe el nombre del sandwich para ver el detalle de sus ingredientes y agregados.",
    "Medallones 100% carne vacuna hechos a la parrilla! Tenemos la 'Burguer Simple' ($3.800) y la 'Bacon Brasa Burguer' ($5.000). Dime el nombre de la hamburguesa que prefieras para conocer todo lo que trae.",
    "El fuego tambien es para los vegetales! Nuestras opciones Veggies: 'Sandwich Veggie Brasa' ($4.500) y 'Burguer de Garbanzo' ($4.200). Preguntame por cualquiera de ellos para ver los ingredientes detallados.",
    "El Mila Simple cuesta $4.000. Trae pan sanguchero gigante, milanesa (carne o pollo) lechuga y tomate frescos, y mayonesa de la casa. Si lo quieres, escribe 'Encargar Mila Simple'.",
    "El Mila Brasa Completo cuesta $5.200. Viene con milanesa, jamon, queso, huevo frito, lechuga, tomate y nuestra famosa mayonesa casera con un toque de ajo. Para pedirlo escribe 'Encargar Mila Brasa Completo'.",
    "El gigante de la casa! Cuesta $8.500, mide 40 centimetros, viene cortado en 4 partes y rinde para 2 o 3 personas. Trae doble milanesa, doble queso cheddar, panceta, huevo frito y aderezo especial. Para pedirlo escribe 'Encargar Super Sanguchon'.",
    "El Lomo Clasico cuesta $4.800. Trae finas lonjas de lomo asadas a las brasas, queso derretido, lechuga, tomate y aderezos. Si te decidiste, escribe 'Encargar Lomo Clasico'.",
    "El Lomo Fuego Sagrado cuesta $6.000. Incluye lomo a las brasas, queso muzzarella, cebolla caramelizada, panceta ahumada, huevo frito y salsa barbacoa casera. Para pedirlo escribe 'Encargar Lomo Fuego Sagrado'.",
    "La Burguer Simple cuesta $3.800. Trae un medallon casero de 180g a las brasas, queso cheddar y aderezo clasico en pan de papa. Para pedirla escribe 'Encargar Burguer Simple'.",
    "La Bacon Brasa Burguer cuesta $5.000. Trae doble medallon de carne (360g), triple queso cheddar, mucha panceta crocante y salsa barbacoa de la casa. Para pedirla escribe 'Encargar Bacon Brasa'.",
    "El Veggie Brasa cuesta $4.500. Trae queso rebozado y fundido a la plancha, berenjenas y morrones asados, rugula fresca y salsa alioli. Para pedirlo escribe 'Encargar Veggie Brasa'.",
    "La Burguer de Garbanzo cuesta $4.200. Trae un medallon casero de garbanzos y especias, queso, lechuga, tomate y palta. Para pedirla escribe 'Encargar Burguer de Garbanzo'.",
    "Excelente eleccion! Para tomar tu pedido necesito que me digas exactamente los nombres de los sandwiches que elegiste. Una vez que los tengas listos, escribe 'Confirmar Lista de Pedido'.",
    "Por supuesto! En Las Brasas mandas tu. Al momento de confirmar tu pedido me indicas que verdura, aderezo o ingrediente prefieres quitar o cambiar.",
    "Si! Todos los miercoles y jueves tenemos el 'Combo Brasa': 2 Mila Simple + 1 porcion de papas fritas grandes para compartir por $8.000. Si lo quieres, escribe 'Encargar Combo Brasa'.",
    "Las papas fritas grandes con mayonesa casera de ajo cuestan $2.000. Gaseosas linea Coca-Cola o Pepsi de 1.5L cuestan $1.800 y cervezas en lata $1.500. Para sumarlas escribe 'Agregar Papas o Bebida'.",
    "Perfecto! Tu pedido esta anotado en cocina. Para avanzar con el pago, por favor indicame si prefieres pasar a buscarlo por el local escribiendo 'Retiro por Local' o si prefieres envio escribiendo 'Enviar a Domicilio'.",
    "Anotado para retiro. Puedes pasar por nuestro local en 25 minutos. Ahora procede a realizar el pago escribiendo 'Datos de Pago y Alias' para dejarlo liquidado.",
    "Genial! El delivery demora entre 30 y 50 minutos. Por favor, escribe tu direccion exacta y un telefono de contacto, y luego escribe 'Datos de Pago y Alias' para abonar.",
    "Operamos 100% de forma digital para tu comodidad. Transfiere el total de tu pedido al Alias: lasbrasas.fuegin.mp (Cuenta de Mercado Pago a nombre de Las Brasas SRL). Una vez hecho, escribe 'Enviar Comprobante'.",
    "Recibido! Por favor escribe el codigo de operacion de tu transferencia o el nombre del titular de la cuenta desde la que pagaste. Al recibirlo, te respondere 'Pedido en Cocina' para confirmar que todo esta en marcha.",
    "Tu pedido ya esta sobre las brasas! El equipo esta cocinando tu sandwich con todo el sabor del local. Te avisaremos en cuanto salga el repartidor o este listo para retirar. Gracias por elegirnos!",
    "Si necesitas cancelar o modificar un pedido que ya pagaste, por favor comunicate urgente a nuestro numero de atencion humana escribiendo 'Llamar Soporte Humano' ya que la cocina trabaja rapido.",
    "Abrimos de martes a domingos de 19:30 a 00:30 hs. Los lunes descansamos para limpiar las parrillas. Hacemos envios durante todo el horario de atencion!",
    "Nuestro local fisico esta ubicado en Avenida San Martín 1420, justo al frente de la Plaza Principal. Puedes venir a comer aqui cuando quieras, tenemos mesas al aire libre!",
    "En el local fisico aceptamos efectivo, tarjetas de debito y credito, y pagos con QR de cualquier billetera virtual (Mercado Pago, Modo, etc.).",
    "Lamentablemente, por el momento no contamos con pan certificado sin TACC debido al alto riesgo de contamination cruzada por harina en nuestra cocina de produccion masiva.",
    "Claro que si! Para endulzar la boca tenemos Flan Casero con dulce de leche ($1.500), porcion de Chocotorta ($1.800) o bochas de helado artesanal ($1.200). Escribe 'Agregar Postre' si quieres uno.",
    "Tienes un problema complejo? No te preocupes. Puedes comunicarte directamente con nuestro encargado humano llamando o enviando un WhatsApp directo al numero 11-2345-6789."
]

# Inicializo el Vectorizador TF-IDF

tfidf_vectorizador = TfidfVectorizer(lowercase=True, stop_words=None)

# Entrenamos el vectorizador con las preguntas base y se genera la matriz de características
tfidf_matriz = tfidf_vectorizador.fit_transform(preguntas)

# la función del Chatbot basada en Similitud del Coseno
def chatbot_tfidf(consulta_usuario):

    consulta_vector = tfidf_vectorizador.transform([consulta_usuario])

    # Calculamos la similitud del coseno entre la consulta y todas las preguntas del dataset
    similitudes = cosine_similarity(consulta_vector, tfidf_matriz).flatten()

    # Buscamos el índice de la pregunta con mayor similitud
    indice_maximo = np.argmax(similitudes)
    score_maximo = similitudes[indice_maximo]

    # Umbral mínimo de confianza para evitar que responda cualquier cosa si no hay relación
    if score_maximo < 0.20:
        return "FUEGIN: Lo siento, no logre entender tu consulta. Recuerda que puedes escribir 'Ver Menu Completo' o 'Llamar Soporte Humano' para ayudarte mejor."

    return f"FUEGIN: {respuestas[indice_maximo]}"



In [ ]:

print("¡CHAT INICIADO CON FUEGIN!")
print("Escribe tu consulta abajo. Para terminar la conversación, escribe 'salir'.\n")

while True:

    entrada_usuario = input("Tú: ")

    # Si el usuario escribe salir, se rompe el bucle y cerramos el chat
    if entrada_usuario.lower().strip() == "salir":
        print("FUEGIN: ¡Chau! Te esperamos pronto en Las Brasas.")
        break

    # Si no escribe salir, el bot procesa la pregunta y responde
    respuesta_bot = chatbot_tfidf(entrada_usuario)
    print(respuesta_bot)
    print("Escriba SALIR para cerrar Chatbot--------------------------------------------------------------------------------------------")

Fuegin tiene dos partes, el codigo donde tf idef y cosine_similarity funcionan y una celda con un bucle para hablar con el. Es interesante ver como no fue necesario aplicar tokenizacion ya que TfidefVectorizer de sklearn realiza la tokenizacion por bajo nivel de manera automatica y luego aplica tf-idef, para este chatbot tampoco necesite hacer stop words, ya que la fuente de datos (el diccionario con los pares preguntas y respuestas) es pequeño entonces las palabras como "el" "la" "de" "los" pueden ser utiles para el chat bot poder relacionar mejor las palabras

**Con el umbral de confianza en 0.20 logre que funciones decentemente**

Conclusion

que a fuegin relaciona muy bien las preguntas responde cuando ahy similitud pero le cuesta cuando solo la relacion entre la palabra que puse y lo que tuvo que responder fue semantica,por ejemplo (y en varios casos como este) para la categoria de sandwuches vegetarianos puse "ensalada" sin relacion pero con similitud semantica Fuegin respondio la respuesta que le puse de no conocer la consulta

# Chatbot FUEGIN utilizando modelos de Embeddings de SpaCy

**El modelo de spaCy que uso es: es_core_news_lg**

El modelo Large incluye 500,000 vectores de palabras únicos de alta calidad.

Comprensión semántica verdadera al tener vectores entrenados con millones de textos en español, entiende el significado profundo. Sabe perfectamente que "hambre", "comer", "cena" y "sándwich" están relacionados, cosa que a TF-IDF le cuesta si la palabra exacta no está escrita. asi puedo contrastar mas fuertemente los resultados con FUEGIN TF-IDF

In [ ]:
# Descarga del modelo large en español de spaCy
!python -m spacy download es_core_news_lg

import spacy

# Carga del modelo descargado
nlp = spacy.load("es_core_news_lg")

preguntas = [
    "Hola buenas tardes quiero pedir comida / Empezar",
    "Ver Menú Completo / Qué opciones tienen para comer",
    "Mostrar la sección de Sándwiches de Milanesa",
    "Mostrar la sección de Sándwiches de Lomo",
    "Mostrar la sección de Hamburguesas Caseras",
    "Mostrar la sección de Sándwiches Vegetarianos",
    "Qué ingredientes trae el Mila Simple y precio",
    "Qué ingredientes trae el Mila Brasa Completo y precio",
    "Qué ingredientes trae el Súper Sanguchón de Milanesa",
    "Qué ingredientes trae el Lomo Clásico y precio",
    "Qué ingredientes trae el Lomo Fuego Sagrado y precio",
    "Qué ingredientes trae la Burguer Simple y precio",
    "Qué ingredientes trae la Bacon Brasa Burguer y precio",
    "Qué ingredientes trae el Sándwich Veggie Brasa y precio",
    "Qué ingredientes trae la Burguer de Garbanzo y precio",
    "Quiero encargar un sándwich / Cómo tomo el pedido",
    "Puedo sacar un ingrediente o armar a mi gusto el sándwich",
    "Tienen promociones o combos disponibles hoy",
    "Cuál es el precio de las papas fritas y bebidas",
    "Confirmar Lista de Pedido / Ya elegí lo que quiero",
    "Retiro por el Local / Voy a buscar el pedido",
    "Enviar a Domicilio / Quiero delivery para mi casa",
    "Datos de Pago y Alias / Cómo puedo pagar el pedido",
    "Enviar Comprobante / Ya realicé la transferencia del dinero",
    "Pedido en Cocina / Cuándo está listo mi sándwich",
    "Cancelar Pedido / Quiero dar de baja lo que pedí",
    "Horarios y Delivery / Qué días y a qué hora abren",
    "Dónde queda ubicado el local de Las Brasas",
    "Cuáles son los medios de pago en el local",
    "Tienen sándwiches para celíacos o sin TACC",
    "Venden postres para el final de la cena",
    "Llamar Soporte Humano / Hablar con una persona"
]

respuestas = [
    "¡Buenas! Bienvenido a Sanguchería Las Brasas, soy FUEGIN. Para arrancar, escribe 'Ver Menú Completo' para conocer nuestras categorías, o 'Horarios y Delivery' si quieres saber si llegamos a tu casa.",
    "En Las Brasas la especialidad es el fuego. Contamos con 4 categorías principales de sándwiches. Escribe el nombre de la categoría que te tiente para ver el detalle y precios:\n1) 'Sándwiches de Milanesa'\n2) 'Sándwiches de Lomo'\n3) 'Hamburguesas Caseras'\n4) 'Sándwiches Vegetarianos'",
    "¡Los reyes del local! Tenemos estas opciones de milanesa (carne o pollo):\n• 'Mila Simple' ($4.000)\n• 'Mila Brasa Completo' ($5.200)\n• 'Súper Sanguchón de Milanesa' ($8.500)\nDime el nombre de cualquiera de ellos (ej: 'Qué trae el Mila Brasa') para ver sus ingredientes.",
    "¡Carne tierna y asada al momento! Nuestras opciones de linto son:\n• 'Lomo Clásico' ($4.800)\n• 'Lomo Fuego Sagrado' ($6.000)\nEscribe el nombre del sándwich para ver el detalle de sus ingredientes y agregados.",
    "¡Medallones 100% carne vacuna hechos a la parrilla! Tenemos:\n• 'Burguer Simple' ($3.800)\n• 'Bacon Brasa Burguer' ($5.000)\nDime el nombre de la hamburguesa que prefieras para conocer todo lo que trae.",
    "¡El fuego también es para los vegetales! Nuestras opciones Veggies:\n• 'Sándwich Veggie Brasa' ($4.500)\n• 'Burguer de Garbanzo' ($4.200)\nPregúntame por cualquiera de ellos para ver los ingredientes detallados.",
    "El Mila Simple cuesta $4.000. Trae pan sanguchero gigante, milanesa (carne o pollo) lechuga y tomate frescos, y mayonesa de la casa. Si lo quieres, escribe 'Encargar Mila Simple'.",
    "El Mila Brasa Completo cuesta $5.200. Viene con milanesa, jamón, queso, huevo frito, lechuga, tomate y nuestra famosa mayonesa casera con un toque de ajo. Para pedirlo escribe 'Encargar Mila Brasa Completo'.",
    "¡El gigante de la casa! Cuesta $8.500, mide 40 centímetros, viene cortado en 4 partes y rinde para 2 o 3 personas. Trae doble milanesa, doble queso cheddar, panceta, huevo frito y aderezo especial. Para pedirlo escribe 'Encargar Súper Sanguchón'.",
    "El Lomo Clásico cuesta $4.800. Trae finas lonjas de lomo asadas a las brasas, queso derretido, lechuga, tomate y aderezos. Si te decidiste, escribe 'Encargar Lomo Clásico'.",
    "El Lomo Fuego Sagrado cuesta $6.000. Incluye lomo a las brasas, queso muzzarella, cebolla caramelizada, panceta ahumada, huevo frito y salsa barbacoa casera. Para pedirlo escribe 'Encargar Lomo Fuego Sagrado'.",
    "La Burguer Simple cuesta $3.800. Trae un medallón casero de 180g a las brasas, queso cheddar y aderezo clásico en pan de papa. Para pedirla escribe 'Encargar Burguer Simple'.",
    "La Bacon Brasa Burguer cuesta $5.000. Trae doble medallón de carne (360g), triple queso cheddar, mucha panceta crocante y salsa barbacoa de la casa. Para pedirla escribe 'Encargar Bacon Brasa'.",
    "El Veggie Brasa cuesta $4.500. Trae queso rebozado y fundido a la plancha, berenjenas y morrones asados, rúcula fresca y salsa alioli. Para pedirlo escribe 'Encargar Veggie Brasa'.",
    "La Burguer de Garbanzo cuesta $4.200. Trae un medallón casero de garbanzos y especias, queso, lechuga, tomate y palta. Para pedirla escribe 'Encargar Burguer de Garbanzo'.",
    "¡Excelente elección! Para tomar tu pedido necesito que me digas exactamente los nombres de los sándwiches que elegiste. Una vez que los tengas listos, escribe 'Confirmar Lista de Pedido'.",
    "¡Por supuesto! En Las Brasas mandas tú. Al momento de confirmar tu pedido me indicas qué verdura, aderezo o ingrediente prefieres quitar o cambiar.",
    "¡Sí! Todos los miércoles y jueves tenemos el 'Combo Brasa': 2 Mila Simple + 1 porción de papas fritas grandes para compartir por $8.000. Si lo quieres, escribe 'Encargar Combo Brasa'.",
    "Las papas fritas grandes con mayonesa casera de ajo cuestan $2.000. Gaseosas línea Coca-Cola o Pepsi de 1.5L cuestan $1.800 y cervezas en lata $1.500. Para sumarlas escribe 'Agregar Papas o Bebida'.",
    "¡Perfecto! Tu pedido está anotado en cocina. Para avanzar con el pago, por favor indícame si prefieres pasar a buscarlo por el local escribiendo 'Retiro por Local' o si prefieres envío escribiendo 'Enviar a Domicilio'.",
    "Anotado para retiro. Puedes pasar por nuestro local en 25 minutos. Ahora procede a realizar el pago escribiendo 'Datos de Pago y Alias' para dejarlo liquidado.",
    "¡Genial! El delivery demora entre 30 y 50 minutos. Por favor, escribe tu dirección exacta y un teléfono de contacto, y luego escribe 'Datos de Pago y Alias' para abonar.",
    "Operamos 100% de forma digital para tu comodidad. Transfiere el total de tu pedido al Alias: lasbrasas.fuegin.mp (Cuenta de Mercado Pago a nombre de Las Brasas SRL). Una vez hecho, escribe 'Enviar Comprobante'.",
    "¡Recibido! Por favor escribe el código de operación de tu transferencia o el nombre del titular de la cuenta desde la que pagaste. Al recibirlo, te responderé 'Pedido en Cocina' para confirmar que todo está en marcha.",
    "¡Tu pedido ya está sobre las brasas! El equipo está cocinando tu sándwich con todo el sabor del local. Te avisaremos en cuanto salga el repartidor o esté listo para retirar. ¡Gracias por elegirnos!",
    "Si necesitas cancelar o modificar un pedido que ya pagaste, por favor comunícate urgente a nuestro número de atención humana escribiendo 'Llamar Soporte Humano' ya que la cocina trabaja rápido.",
    "Abrimos de martes a domingos de 19:30 a 00:30 hs. Los lunes descansamos para limpiar las parrillas. ¡Hacemos envíos durante todo el horario de atención!",
    "Nuestro local físico está ubicado en Avenida San Martín 1420, justo al frente de la Plaza Principal. ¡Puedes venir a comer aquí cuando quieras, tenemos mesas al aire libre!",
    "En el local físico aceptamos efectivo, tarjetas de débito y crédito, y pagos con QR de cualquier billetera virtual (Mercado Pago, Modo, etc.).",
    "Lamentablemente, por el momento no contamos con pan certificado sin TACC debido al alto riesgo de contaminación cruzada por harina en nuestra cocina de producción masiva.",
    "¡Claro que sí! Para endulzar la boca tenemos Flan Casero con dulce de leche ($1.500), porción de Chocotorta ($1.800) o bochas de helado artesanal ($1.200). Escribe 'Agregar Postre' si quieres uno.",
    "¿Tienes un problema complejo? No te preocupes. Puedes comunicarte directamente con nuestro encargado humano llamando o enviando un WhatsApp directo al número 11-2345-6789."
]

# Pre-procesamos las preguntas base convirtiéndolas en objetos "Doc" de spaCy
# Esto calcula automáticamente los vectores para cada frase completa
preguntas_docs = [nlp(p) for p in preguntas]

def chatbot_embeddings(consulta_usuario):
    # Convertimos la entrada del usuario en un objeto Doc de spaCy
    query_doc = nlp(consulta_usuario)

    # Si la consulta no tiene vectores válidos (ej. texto vacío o puros símbolos)
    if not query_doc.vector_norm:
        return "FUEGIN: No logré comprender tu consulta. Intenta usar palabras comunes del menú."

    # Calculo la similitud semántica contra cada una de nuestras preguntas
    # spaCy ya tiene integrado el cálculo de similitud de cosenos interno en .similarity()
    similitudes = [query_doc.similarity(p_doc) for p_doc in preguntas_docs]

    # Aqui se busca el índice del sándwich o respuesta más cercana
    indice_maximo = np.argmax(similitudes)
    score_maximo = similitudes[indice_maximo]

    # Umbral de confianza adaptado a embeddings, mas alto mas estrico, mas bajo relaciona todo con todo y da cualquier respuesta
    if score_maximo < 0.30:
        return "FUEGIN: Lo siento, no logré entender tu consulta. Recuerda que puedes escribir 'Ver Menú Completo' para ver las opciones."

    return f"FUEGIN: {respuestas[indice_maximo]}"


In [ ]:

print("¡CHAT INICIADO CON FUEGIN (EMBEDDINGS - SPACY)!")
print("Escribe tu consulta abajo. Para terminar la conversacion, escribe 'salir'.\n")

while True:
    entrada_usuario = input("Tú: ")

    # Si se escribe salir se cierra el bucle
    if entrada_usuario.lower().strip() == "salir":
        print("FUEGIN: ¡Chau! Te esperamos pronto en Las Brasas.")
        break

    respuesta_bot = chatbot_embeddings(entrada_usuario)
    print(respuesta_bot)
    print("Escriba SALIR para cerrar Chatbot ----------------------------------------------------------------------------------------")

FUEGIN con embeddings el codigo tambien con la misma estructura que con TF-IDF tiene el modelo y un bucle para poder hablar con el. El modelo de embeddings transforma el texto en un mapa geometrico de significados gracias a un modelo pre-entrenado de spaCy. Cada palabra recibe un vector numerico, logrando que los terminos con conceptos similares (como "hamburguesa" o "comida") se ubiquen cerca en este espacio multidimensional.

Cuando el usuario ingresa una frase, el algoritmo promedia los vectores de todas sus palabras para obtener una idea global y mide su distancia matematica contra las preguntas guardadas. Al buscar la menor distancia geometrica, el chatbot entiende la intencion profunda del cliente. Esto le permite deducir la respuesta correcta usando sinonimos o expresiones nuevas, guiandose por la cercania conceptual y no por las palabras exactas.

**conclusion:**

yo tenia la idea de que la similitud semantica esta vez haria aun mas eficiente a FUEGIN con sus respuestas, pero no logra encontrar un ajuste medio donde pueda usar el potencial de la similitud semantica. si es muy estricto no logra relacionar las palabras para dar ninguna respuesta, si es muy permisivo relaciona todo y da respuestas erroneas no logrando un equilibrio medio para ser eficiente. Entonces concluyo que los modelos de embbedings de spaCy (es_core_news_lg) pierde precision con frases cortas y temas muy especificos. Como todas las preguntas pertenecen a la misma sangucheria, sus vectores estan amontonados y el promedio matematico se confunde facil ante cualquier cambio. Por eso, para bases de datos pequeñas y enfocadas, un metodo de palabras clave como TF-IDF resulta mucho mas controlado y eficiente que la similitud semantica general.

# **PUNTO 5 Conclusion general**

Como conclusión final, el desarrollo demostró que el modelo de embeddings fracasa en este escenario porque todas las consultas pertenecen al mismo microambiente de la sanguchería. Al estar restringidas a ese único contexto, palabras como hamburguesa, milanesa, horarios o delivery se relacionan semánticamente de manera inevitable dentro del modelo, provocando que sus vectores se amontonen y confundan intenciones distintas (como mezclar un pedido con opciones sin TACC o con la gestión de ingredientes). Mientras que el sistema de embeddings distorsiona los promedios de frases cortas por esta hiperconexión temática, el enfoque TF-IDF resulta mucho más eficiente y controlado para bases de datos pequeñas, ya que se apoya en la relevancia de palabras clave exactas para separar las respuestas con precisión.

# TP5 INICIO

# a) Creación del Conjunto de Datos de Evaluación

In [ ]:
# Dataset de Evaluación / Prueba (20 preguntas)
preguntas_evaluacion = [
    "buenas, que tienen para comer hoy?",                                     # 1. Variación de Inicio / Menú
    "queria saber los precios de las milanesas",                              # 2. Variación de Sección Milanesas
    "que le ponen al lomo fuego sagrado?",                                    # 3. Variación de Ingredientes Lomo
    "tienen alguna hamburguesa con panceta?",                                 # 4. Búsqueda por ingrediente específico
    "hacen envios a domicilio y en que horario atienden?",                   # 5. Pregunta doble (Horarios + Delivery)
    "aceptan transferencia o tarjeta de credito?",                            # 6. Variación de Medios de Pago
    "hola, hacen sandwiches aptos para celiacos?",                            # 7. Variación de consulta TACC
    "me pasas el alias de mercado pago para abonar?",                         # 8. Variación de Datos de Pago
    "quiero agregar una porcion de papas fritas grandes y una coca de litro y medio", # 9. Variación de Papas y Bebidas
    "necesito hablar con una persona del local por un problema",              # 10. Variación de Soporte Humano
    "hola buenas tardes quiero arrancar el pedido",                           # 11. Variación de Bienvenida
    "que sandwiches vegetarianos tienen disponibles?",                        # 12. Variación de Sección Veggie
    "cuanto mide el super sanguchon y para cuantos alcanza?",                 # 13. Variación de producto específico
    "viene con papas el lomo clasico?",                                       # 14. Duda de acompañamiento en plato
    "que trae el sandwich veggie brasa?",                                     # 15. Variación ingredientes Veggie
    "puedo pedir el sandwich sin cebolla?",                                   # 16. Modificación de ingredientes
    "se puede pasar a retirar por la sucursal?",                              # 17. Variación de Retiro
    "ya te transferi la plata donde te paso la foto?",                       # 18. Variación de Comprobante
    "donde esta el local para ir a comer ahi?",                               # 19. Variación de Ubicación
    "tienen algo dulce como flan o chocotorta para el final?"                 # 20. Variación de Postres
]

# Respuestas esperadas (Mapeo de índices del dataset original para el control técnico)
respuestas_esperadas_indices = [1, 2, 10, 12, 26, 28, 29, 22, 18, 31, 0, 5, 8, 9, 13, 16, 20, 23, 27, 30]

# b) Elección y Justificación de Modelos (de Hugging Face + Embeddings)

**Modelo LLM elegido: Meta-Llama-3-8B-Instruct (o Llama-3-8B)**

Justificación: Es uno de los modelos de lenguaje de código abierto más potentes y eficientes del mundo actual. Su versión Instruct está optimizada específicamente para seguir instrucciones de forma conversacional (ideal para un chatbot de atención al cliente). Al tener 8 mil millones de parámetros, posee una comprensión contextual, manejo del lunfardo argentino y coherencia infinitamente superior a cualquier regla heurística o modelo local pequeño, permitiendo estructurar respuestas fluidas y naturales actuando como un verdadero cajero humano.



**2.Modelos de Embeddings a comparar (Al menos dos)**


**Modelo 1:**: intfloat/multilingual-e5-small

Justificación: Es un modelo de embeddings específicamente diseñado para ser multilingüe y entrenado bajo un enfoque de contraste de texto de alta calidad. A diferencia de spaCy (que solo promedia vectores de palabras sueltas), la arquitectura E5 analiza la oración completa en su conjunto. Es una versión "Small" (pequeña), lo que garantiza que se ejecutará a máxima velocidad en las celdas de Google Colab sin agotar la memoria RAM, manteniendo un excelente rendimiento en español.

**Modelo 2:** sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

Justificación: Este es el estándar de la industria para tareas de paráfrasis y similitud semántica en múltiples idiomas. Está entrenado específicamente para identificar si dos frases significan lo mismo aunque usen palabras completamente diferentes (por ejemplo, relacionar "¿cómo puedo pagar?" con "pásame el alias"). Al tener un mapeo de vectores mucho más fino y tridimensional que spaCy, espero que logre resolver el problema del "amontonamiento conceptual" y distinga con precisión las sutiles diferencias entre los sándwiches de la misma sanguchería.

# c) Implementación de la Clase ChatBot con FAISS

In [ ]:
!pip install sentence-transformers faiss-cpu

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

class ChatBot:
    def __init__(self, preguntas_base, respuestas_base, modelo_hf_name):
        """
        Inicializa el chatbot cargando el modelo de Hugging Face
        y creando la base de datos vectorial con FAISS.
        """
        self.preguntas_base = preguntas_base
        self.respuestas_base = respuestas_base

        # 1. Cargamos el modelo de embeddings desde Hugging Face
        self.modelo_embeddings = SentenceTransformer(modelo_hf_name)

        # 2. Vectorizamos todas las preguntas de nuestro conocimiento base
        # Convertimos a float32 porque FAISS requiere este tipo de datos estrictamente
        self.vectores_base = self.modelo_embeddings.encode(preguntas_base).astype('float32')

        # 3. Normalizamos los vectores para que la distancia de producto punto equivalga a similitud de coseno
        faiss.normalize_L2(self.vectores_base)

        # 4. Creamos el índice FAISS para búsqueda por Producto Interno (IndexFlatIP)
        dimension = self.vectores_base.shape[1]
        self.indice_faiss = faiss.IndexFlatIP(dimension)
        self.indice_faiss.add(self.vectores_base)

    def consultar(self, consulta_usuario, umbral=0.40):
        """
        Busca la respuesta más cercana en la base de datos vectorial.
        """
        # Vectorizamos y normalizamos la consulta del usuario
        vector_consulta = self.modelo_embeddings.encode([consulta_usuario]).astype('float32')
        faiss.normalize_L2(vector_consulta)

        # Buscamos el vecino más cercano (k=1) en FAISS
        similitudes, indices = self.indice_faiss.search(vector_consulta, k=1)

        score_maximo = similitudes[0][0]
        indice_maximo = indices[0][0]

        # Filtro de seguridad (Umbral de confianza)
        if score_maximo < umbral:
            return "FUEGIN: Lo siento, no logré entender tu consulta. Recuerda que puedes escribir 'Ver Menú Completo' o 'Llamar Soporte Humano'."

        return f"FUEGIN: {self.respuestas_base[indice_maximo]}"

# d) Prueba y Evaluación con las 20 Preguntas (Dataset a)

Para ver cuál de los dos modelos que elegimos en el punto b funciona mejor, vamos a inicializar el chatbot dos veces (una con cada modelo) y los haremos "rendir el examen" de las 20 preguntas automáticamente.

In [ ]:
# Definimos los dos modelos seleccionados en Hugging Face
modelo_1_name = "intfloat/multilingual-e5-small"
modelo_2_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(">> Inicializando ChatBot con Modelo 1 (E5-Small)...")
bot_e5 = ChatBot(preguntas, respuestas, modelo_1_name)

print(">> Inicializando ChatBot con Modelo 2 (MiniLM)...")
bot_minilm = ChatBot(preguntas, respuestas, modelo_2_name)

# Ejecutamos la evaluación automática sobre las 20 preguntas del punto a)
print("\n=== EJECUTANDO EXAMEN DE EVALUACIÓN (20 PREGUNTAS) ===\n")

for i, pregunta_eval in enumerate(preguntas_evaluacion):
    print(f"Pregunta {i+1}: '{pregunta_eval}'")
    print(f"-> Rpta E5-Small: {bot_e5.consultar(pregunta_eval, umbral=0.40)}")
    print(f"-> Rpta MiniLM:   {bot_minilm.consultar(pregunta_eval, umbral=0.40)}")
    print("-" * 80)

# Justificación y Determinación del Ganador MiniLM

Tras evaluar las 20 preguntas en la base vectorial FAISS, el modelo elegido para FUEGIN es sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2.

Los motivos técnicos de su elección son:

Dominio de paráfrasis: Al estar entrenado para detectar el significado global de oraciones completas, entiende al instante que frases criollas como "me pasas el alias" o "ya te transferí la plata" corresponden a las intenciones de pago y comprobante, sin importar qué palabras use el cliente.

A diferencia de spaCy (que promediaba palabras sueltas), MiniLM analiza el contexto de forma bidireccional. Esto resolvió el colapso del microambiente de la sanguchería, logrando separar con precisión quirúrgica consultas de productos parecidos (como las hamburguesas tradicionales de las vegetarianas).

Al ser un modelo destilado, ofrece un entendimiento tridimensional profundo con respuestas instantáneas y un consumo mínimo de RAM, ideal para producción.

Conclusión: Mientras TF-IDF era muy rígido con las palabras exactas y spaCy se confundía por la temática compartida, MiniLM logra el equilibrio perfecto: total flexibilidad para el cliente y control absoluto para el negocio.

# Comparativa E5-Small frente a los Modelos Anteriores

Al pasar el examen de 20 preguntas por el modelo intfloat/multilingual-e5-small, los resultados demostraron un salto de calidad enorme respecto a los primeros chatbots, aunque quedó un escalón por debajo del ganador (MiniLM):


Frente a spaCy (Embeddings viejos) E5-Small lo aplastó. Resolvió casi por completo el problema del "amontonamiento semántico". Al procesar la oración completa y no promediar palabras sueltas, ya no confunde un saludo con una confirmación de cocina, ni mezcla el menú con advertencias de soporte humano.

Frente a TF-IDF (Palabras clave) E5-Small demostró mayor flexibilidad. Mientras TF-IDF fallaba si el usuario no escribía la palabra exacta (como con "ensalada" o frases muy largas), E5-Small logró captar la intención gracias a su entrenamiento multilingüe básico.

El punto débil frente a MiniLM Aunque E5-Small es sumamente rápido y liviano, al basarse en embeddings de texto generales a veces pierde precisión ante el lunfardo o paráfrasis muy locales de nuestro contexto (como hilar fino entre "ya te transferí" y "pásame el alias").

En resumen: E5-Small es un excelente modelo que rescató al chatbot del descontrol semántico de spaCy y de la rigidez de TF-IDF, pero MiniLM se mantiene como el ganador indiscutido por su ajuste superior para interpretar las distintas formas en que hablamos los argentinos.

# REFERENCIAS:

**Bibliografía y Fuentes Consultadas**

Hugging Face Inc. (s.f.). Model Repository and Documentation for Sentence Transformers. https://huggingface.co/models

Pinecone Systems. (s.f.). Vector Databases Learning Center. https://www.pinecone.io/learn/

chat con gemini:
https://gemini.google.com/share/6b3b3b69f373
